<a href="https://colab.research.google.com/github/AlessandroRestagno/Advanced-lane-finding-P4-Udacity-Self-Drivng-Car/blob/master/Termotecnica_Casa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initial values

In [ ]:
lenght = 17.66
width = 11.66
height = 2.8

window_area = 30
door_area = 2

U_window = 1.
U_wall = 0.4
U_door = 1.
U_floor = 0.15
U_ceiling = 0.4
rho = 0.33
K_vent = 0.7
t_max = 20
t_min = -5
COP = 3.5
K_thermal_bridge = 0.05
wall_area = 2 * (lenght + width) * height - door_area - window_area
area = 140.
volume = area * height

t_target = 20.


# Transmittance (W/mq*K)

In [ ]:
H_tr = window_area * U_window + wall_area * U_wall + door_area * U_door + area * U_floor + area * U_ceiling
H_vent = volume * rho * K_vent
H_tot = H_tr + H_vent

# Hourly Energy Consumption Formula

In [ ]:
def hourly_energy_consumption(t_target,t_outside,transmittance,coeff_bridge,actual_COP):
  delta_t = max((t_target - t_outside),0)
  thermal_power = (transmittance * delta_t) * (1 + coeff_bridge)
  electric_power = thermal_power / actual_COP
  return electric_power


## Get temperature values for Montanera

In [ ]:
!pip install meteostat -q

In [ ]:
from meteostat import Stations, Hourly
from datetime import datetime
import numpy as np

station_id = '16114'   # Example: Mondovi

# --- Define your period ---
start = datetime(2025, 10, 1)
end   = datetime(2025, 10, 31)

# --- 1️⃣ Hourly data ---
hourly_data = Hourly(station_id, start, end)
hourly_df = hourly_data.fetch()

print(hourly_df.head(1))

            temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt    pres  tsun  \
time                                                                        
2025-10-01  12.3  10.2  87.0   0.2  <NA>  251.0   7.9  <NA>  1021.6  <NA>   

            coco  
time              
2025-10-01  17.0  


## Calculate Energy expenditure and cost

In [ ]:
hourly_df['DayConsumption'] = hourly_df.apply(
    lambda row: hourly_energy_consumption(
        t_target, # set temperature inside house
        row['temp'],
        H_tot,
        K_thermal_bridge,
        COP
    ),
    axis=1
)

print('Total Energy consumption for this month is',np.round(hourly_df['DayConsumption'].sum()/1000,0),'kW')
print('Total expenditure for this month is EUR',np.round(hourly_df['DayConsumption'].sum()*0.35/1000,2))


Total Energy consumption for this month is 414.0 kW
Total expenditure for this month is EUR 145.01


## Legacy code

In [ ]:
delta_t = abs(t_max - t_min)
thermal_power = (H_tot * delta_t) * (1 + K_thermal_bridge)
electric_power = thermal_power / COP

In [ ]:
print("It needs ",int(electric_power),"W at its peak")

It needs  3763 W at its peak


In [ ]:
import requests
import pandas as pd

city = "Mondovi, Italy"
year = 2025

# 1) geocode city -> lat, lon
geo_url = "https://geocoding-api.open-meteo.com/v1/search"
geo_params = {
    "name": city,
    "count": 1,
    "language": "en",
    "format": "json"
}
geo_resp = requests.get(geo_url, params=geo_params)
geo_data = geo_resp.json()
lat = geo_data["results"][0]["latitude"]
lon = geo_data["results"][0]["longitude"]

# 2) download daily data for that year
start_date = f"{year}-08-01"
end_date   = f"{year}-08-31"

weather_url = "https://api.open-meteo.com/v1/forecast"
weather_params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_date,
    "end_date": end_date,
    "daily": ["temperature_2m_max", "temperature_2m_min"],
    "timezone": "auto"
}
weather_resp = requests.get(weather_url, params=weather_params)
weather_data = weather_resp.json()

# 3) put into pandas
df = pd.DataFrame({
    "date": weather_data["daily"]["time"],
    "tmax": weather_data["daily"]["temperature_2m_max"],
    "tmin": weather_data["daily"]["temperature_2m_min"],
})

# 4) compute yearly min/max
year_max = df["tmax"].max()
year_min = df["tmin"].min()

print(f"City: {city}")
print(f"Year: {year}")
print(f"Maximum temperature in the year: {year_max} °C")
print(f"Minimum temperature in the year: {year_min} °C")


KeyError: 'daily'

In [ ]:
import numpy as np

def hourly_temperature(Tmin, Tmax, sunrise=7, Tmax_time=15):
    hours = np.arange(0, 24)
    temps = np.zeros(24)

    # Daytime (sunrise to Tmax_time)
    for h in range(int(sunrise), int(Tmax_time) + 1):
        temps[h] = Tmin + (Tmax - Tmin) * np.sin(
            np.pi * (h - sunrise) / (2 * (Tmax_time - sunrise))
        )**2

    # Nighttime (Tmax_time to next sunrise)
    for h in range(int(Tmax_time) + 1, (24+sunrise)):
      if h < 24:
        temps[h] = Tmin + (Tmax - Tmin) * np.cos(
            np.pi * (h - Tmax_time) / (2 * (24 - Tmax_time + sunrise))
        )**2
      else:
        temps[h-24] = Tmin + (Tmax - Tmin) * np.cos(
            np.pi * (h - Tmax_time) / (2 * (24 - Tmax_time + sunrise))
        )**2

    return pd.Series(temps, index=hours, name="Temp (°C)")

def daily_energy_consumption(Tmin, Tmax, t_target, H_tot, K_thermal_bridge, COP):
    # 1. Get estimated hourly temps
    hourly_temps = hourly_temperature(Tmin, Tmax)

    # 2. Compute hourly consumption for each hour
    hourly_values = [
        hourly_energy_consumption(t_target, T, H_tot, K_thermal_bridge, COP)
        for T in hourly_temps
    ]

    # 3. Sum all 24 hours
    return sum(hourly_values)